# Explore the cleaned FAERS data in DuckDB

Use this notebook to open the verified Stage 2 database and learn its tables, columns, data types, row grain, relationships, and common values. Run cells from top to bottom with **Shift+Enter**, and change the small exploration settings where indicated.

**Read-only:** this notebook connects to DuckDB in read-only mode. It is for inspection and query experiments; it does not change the database. Start with the `.venv` kernel selected.

The five dashboard wireframes are available in `FAERS Dashboard Wireframes.pdf`. Later sections map exploratory queries to the Safety Landscape, Signal Workbench, Drug Profile, Patient Population, and Reporting/Data Health dashboards. Some proposed metrics need additional mapping or signal-analysis work; the notebook calls those out rather than inventing them.

## 1. Find and open the Stage 2 database

The default is `outputs/stage2_clean/faers_clean.duckdb`. If you move the database, edit `DB_PATH` in the next cell. The database is local and excluded from Git; a new clone must first create the Stage 1 and Stage 2 outputs. The connection is opened with `read_only=True` to protect the file.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_DIR = next(
    (p for p in (here, *here.parents)
     if (p / "Data pipeline code" / "build_stage2_clean.py").is_file() and (p / ".git").exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the project folder."
    )

# Change this path if your verified database is stored elsewhere.
DB_PATH = PROJECT_DIR / "outputs" / "stage2_clean" / "faers_clean.duckdb"
if not DB_PATH.is_file():
    raise FileNotFoundError(
        f"Clean database not found: {DB_PATH}\n"
        "Set DB_PATH to the location of your verified faers_clean.duckdb file."
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f"Connected read-only to: {DB_PATH}")
print(f"DuckDB version: {duckdb.__version__}")

## 2. What is inside this database?

`clean_*` tables are the cleaned analysis layer. `audit_*` tables preserve validation and coding summaries; `clean_meta` records the rules version and build metadata. The row counts below are exact counts read from the database, not estimates.

In [ ]:
table_catalog = con.execute("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    WHERE table_catalog = current_database()
    ORDER BY table_schema, table_name
""").df()
display(table_catalog)

table_names = table_catalog.loc[
    table_catalog["table_schema"] == "main", "table_name"
].tolist()
table_counts = []
for table_name in table_names:
    row_count = con.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
    table_counts.append({"table": table_name, "rows": row_count})
display(pd.DataFrame(table_counts).sort_values("table").reset_index(drop=True))

## 3. Read the build metadata

These values identify which rules produced this database and the date range it covers. `source_quarter` on a case is the quarter file that supplied its selected latest version; `first_received_quarter` is the appropriate cohort field for new-report trends.

In [ ]:
display(con.execute("SELECT key, value FROM clean_meta ORDER BY key").df())

print("Case counts by first-received cohort:")
display(con.execute("""
    SELECT first_received_quarter, first_received_cohort,
           count(*) AS cases,
           count(*) FILTER (WHERE is_provisional) AS provisional_cases
    FROM clean_case
    GROUP BY first_received_quarter, first_received_cohort
    ORDER BY min(first_received_quarter_start) NULLS FIRST
""").df())

## 4. Explore column names and data types

DuckDB stores parsed dates as `DATE`, numeric measurements as numeric types, and source/coded text as `VARCHAR`. The query below lists every column and type for the selected table. Change `TABLE_TO_INSPECT` to another table name from the catalog.

In [ ]:
TABLE_TO_INSPECT = "clean_case"  # Try clean_drug, clean_dose, clean_reaction, clean_outcome, clean_onset, clean_indication, clean_report_source.
if TABLE_TO_INSPECT not in table_names:
    raise ValueError(f"Unknown table {TABLE_TO_INSPECT!r}. Choose a table from the catalog above.")

column_info = con.execute("""
    SELECT ordinal_position, column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_catalog = current_database() AND table_schema = 'main'
      AND table_name = ?
    ORDER BY ordinal_position
""", [TABLE_TO_INSPECT]).df()
display(column_info)
print(f"{TABLE_TO_INSPECT}: {len(column_info)} columns")

## 5. Grain: what does one row mean?

The case table is one row per selected case. Several other tables are detail tables with multiple rows per case. Knowing the grain before counting or joining prevents accidental over-counting.

In [ ]:
grain_guide = pd.DataFrame([
    ("clean_case", "one row per selected case/report version", "caseid; primaryid is the selected report key"),
    ("clean_drug", "one row per primaryid + drug_seq", "primaryid; drug_seq_raw for drug-level detail"),
    ("clean_dose", "source drug rows with dose or route data", "primaryid + drug_seq_raw; can be multiple per drug"),
    ("clean_reaction", "one row per primaryid + reaction term", "primaryid; multiple reactions per case"),
    ("clean_outcome", "one row per primaryid + outcome code", "primaryid; multiple outcomes per case"),
    ("clean_onset", "selected administered primary-suspect drug per eligible case", "primaryid; onset may be unavailable"),
    ("clean_indication", "one row per primaryid + drug sequence + indication", "primaryid + drug_seq_raw to a drug"),
    ("clean_report_source", "one row per primaryid + source code", "primaryid; not every case has source rows"),
], columns=["table", "one row represents", "key / relationship"])
display(grain_guide)

## 6. Preview a few rows

Use a small limit when exploring large tables. The helper below only accepts table names discovered in this database and quotes the identifier. It does not accept arbitrary SQL.

In [ ]:
def preview_table(table_name, limit=8):
    if table_name not in table_names:
        raise ValueError(f"Unknown table: {table_name!r}")
    if not isinstance(limit, int) or not 1 <= limit <= 100:
        raise ValueError("limit must be an integer between 1 and 100")
    return con.execute(f'SELECT * FROM "{table_name}" LIMIT {limit}').df()

display(preview_table("clean_case", 5))
display(preview_table("clean_drug", 5))

## 7. Follow one case through the linked tables

Change `SAMPLE_CASEID` to any case ID from a preview. This joins one case to its selected drug rows using `primaryid`, then retrieves reactions and outcomes separately. Expect multiple rows in each child table. Drug-dose and indication detail need the compound key `primaryid` + `drug_seq_raw`.

In [ ]:
SAMPLE_CASEID = con.execute(
    "SELECT caseid FROM clean_case ORDER BY caseid LIMIT 1"
).fetchone()[0]
case_primaryid = con.execute(
    "SELECT primaryid FROM clean_case WHERE caseid = ?", [SAMPLE_CASEID]
).fetchone()[0]
print(f"caseid={SAMPLE_CASEID}; primaryid={case_primaryid}")

print("Case-level row:")
display(con.execute("SELECT * FROM clean_case WHERE caseid = ?", [SAMPLE_CASEID]).df())
print("Drug rows:")
display(con.execute("""
    SELECT drug_seq_raw, role_label, is_administered, ingredient_std,
           ingredient_source, is_combination, n_source_rows
    FROM clean_drug WHERE primaryid = ? ORDER BY drug_seq NULLS LAST, drug_seq_raw
""", [case_primaryid]).df())
print("Reaction rows:")
display(con.execute("""
    SELECT pt, n_source_rows, recurred_on_rechallenge
    FROM clean_reaction WHERE primaryid = ? ORDER BY pt
""", [case_primaryid]).df())
print("Outcome rows:")
display(con.execute("""
    SELECT outcome_code, outcome_label, severity_rank
    FROM clean_outcome WHERE primaryid = ? ORDER BY severity_rank DESC
""", [case_primaryid]).df())
print("Dose and route detail for this case's drugs:")
display(con.execute("""
    SELECT drug_seq_raw, role_cod, route, dose_amt, dose_unit, dose_form, dose_freq
    FROM clean_dose WHERE primaryid = ? LIMIT 20
""", [case_primaryid]).df())

## 8. Why joining all detail tables at once can inflate counts

A case with several drugs, reactions, and outcomes produces a many-to-many row multiplication if all three child tables are joined directly. This cell compares the number of source child rows with the number of rows a direct join would produce for the sample case. For analysis, aggregate each child table to the intended grain first, then join those aggregates to cases.

In [ ]:
display(con.execute("""
    WITH d AS (SELECT count(*) AS n FROM clean_drug WHERE primaryid = ?),
         r AS (SELECT count(*) AS n FROM clean_reaction WHERE primaryid = ?),
         o AS (SELECT count(*) AS n FROM clean_outcome WHERE primaryid = ?),
         joined AS (
           SELECT count(*) AS n
           FROM clean_case c
           LEFT JOIN clean_drug d USING (primaryid)
           LEFT JOIN clean_reaction r USING (primaryid)
           LEFT JOIN clean_outcome o USING (primaryid)
           WHERE c.primaryid = ?
         )
    SELECT d.n AS drug_rows, r.n AS reaction_rows, o.n AS outcome_rows,
           joined.n AS rows_after_direct_join
    FROM d, r, o, joined
""", [case_primaryid, case_primaryid, case_primaryid, case_primaryid]).df())

print("The direct-join row count is not a case count. Count distinct caseid or pre-aggregate child tables.")

## 9. Inspect missingness for one table

Set `TABLE_TO_PROFILE` and run the cell to calculate NULL counts and percentages for every column in that table. This scans the selected table, so start with `clean_case`; larger detail tables may take longer. Blank strings are generally converted to NULL for cleaned fields, while explicit status columns explain missing or rejected values.

In [ ]:
TABLE_TO_PROFILE = "clean_case"
if TABLE_TO_PROFILE not in table_names:
    raise ValueError(f"Unknown table: {TABLE_TO_PROFILE!r}")

profile_columns = con.execute("""
    SELECT column_name FROM information_schema.columns
    WHERE table_catalog = current_database() AND table_schema = 'main'
      AND table_name = ? ORDER BY ordinal_position
""", [TABLE_TO_PROFILE]).fetchall()
quoted_columns = [f'"{row[0].replace(chr(34), chr(34) * 2)}"' for row in profile_columns]
missing_exprs = [
    f'count(*) FILTER (WHERE {column} IS NULL) AS "{row[0]}"'
    for column, row in zip(quoted_columns, profile_columns)
]
missing_counts = con.execute(
    f'SELECT count(*) AS n_total, {", ".join(missing_exprs)} FROM "{TABLE_TO_PROFILE}"'
).fetchone()
n_total = missing_counts[0]
missingness = pd.DataFrame([
    {"column": row[0], "null_rows": missing_counts[index + 1],
     "pct_null": round(100 * missing_counts[index + 1] / n_total, 2) if n_total else 0}
    for index, row in enumerate(profile_columns)
]).sort_values("pct_null", ascending=False).reset_index(drop=True)
print(f"{TABLE_TO_PROFILE}: {n_total:,} rows")
display(missingness)

## 10. Explore a coded field's values

Change the table and column below to inspect another field. The helper verifies the table and column against DuckDB's schema, then returns the most common values. Use cleaned categories for charts (for example `reporter_type` or `age_band`) and keep status/raw fields available when you need to explain data quality.

In [ ]:
TABLE_FOR_VALUES = "clean_case"
COLUMN_FOR_VALUES = "reporter_type"  # Try age_band, sex_std, report_channel, ps_status, event_date_status.
TOP_VALUES = 20

valid_columns = {
    row[0] for row in con.execute("""
        SELECT column_name FROM information_schema.columns
        WHERE table_catalog = current_database() AND table_schema = 'main'
          AND table_name = ?
    """, [TABLE_FOR_VALUES]).fetchall()
}
if TABLE_FOR_VALUES not in table_names or COLUMN_FOR_VALUES not in valid_columns:
    raise ValueError(f"Check table/column: {TABLE_FOR_VALUES}.{COLUMN_FOR_VALUES}")
if not isinstance(TOP_VALUES, int) or not 1 <= TOP_VALUES <= 100:
    raise ValueError("TOP_VALUES must be an integer from 1 to 100")

table_sql = '"' + TABLE_FOR_VALUES.replace('"', '""') + '"'
column_sql = '"' + COLUMN_FOR_VALUES.replace('"', '""') + '"'
display(con.execute(
    f'SELECT {column_sql} AS value, count(*) AS rows '
    f'FROM {table_sql} GROUP BY 1 ORDER BY rows DESC NULLS LAST LIMIT {TOP_VALUES}'
).df())

## 11. Try dashboard-style summaries

These examples illustrate common analysis dimensions. The time series groups on **first-received quarter** and separates provisional cases. The reaction summary counts cases with a reaction rather than raw reaction rows. A FAERS count is a count of reports/cases—not an incidence rate or proof of a drug effect.

In [ ]:
print("Case volume by first-received quarter:")
display(con.execute("""
    SELECT first_received_quarter_start, first_received_quarter,
           is_provisional, count(*) AS cases,
           count(*) FILTER (WHERE has_outcome_code) AS cases_with_outcome
    FROM clean_case
    WHERE first_received_cohort = 'In window'
    GROUP BY 1, 2, 3
    ORDER BY 1
""").df())

print("Most frequently reported reaction terms by distinct cases:")
display(con.execute("""
    SELECT pt, count(DISTINCT caseid) AS cases
    FROM clean_reaction
    GROUP BY pt ORDER BY cases DESC, pt LIMIT 20
""").df())

print("Primary-suspect ingredient counts (top 20; not a causal ranking):")
display(con.execute("""
    SELECT ps_ingredient_std AS ingredient, count(*) AS cases
    FROM clean_case
    WHERE ps_status = 'PS' AND ps_ingredient_std IS NOT NULL
    GROUP BY 1 ORDER BY cases DESC, ingredient LIMIT 20
""").df())

## 12. Try the five dashboard wireframes against the available data

The next cells test representative metrics from the supplied wireframes. These are exploratory checks, not finalized dashboard calculations. A few proposed visuals require data not yet in the clean layer: PRR/chi-square signal statistics need a dedicated signal-analysis step, and `therapeutic_area` needs a reviewed indication-to-area mapping. Duplicate flags support a sensitivity view but do not confirm duplicates. The maturity view below uses `caseversion > 1` as a proxy; exact follow-up rates by initial cohort require case-version history, not just the selected latest version.

### 12a. Safety Landscape: case-level KPI cards

`clean_case` is one row per case, so case counts and shares can be computed without joining child tables. Top-10 drug concentration is based on the selected primary-suspect ingredient and uses the classified PS cases as its denominator.

In [ ]:
landscape_kpis = con.execute("""
    WITH drug_counts AS (
      SELECT ps_ingredient_std AS ingredient, count(*) AS cases
      FROM clean_case
      WHERE ps_status = 'PS' AND ps_ingredient_std IS NOT NULL
      GROUP BY 1
    ), top_drugs AS (
      SELECT ingredient, cases FROM drug_counts
      ORDER BY cases DESC, ingredient LIMIT 10
    ), concentration AS (
      SELECT (SELECT sum(cases) FROM top_drugs) AS top10_cases,
             (SELECT sum(cases) FROM drug_counts) AS all_ps_cases
    )
    SELECT count(*) AS cases,
           count(*) FILTER (WHERE has_outcome_code) AS cases_with_outcome,
           round(100.0 * count(*) FILTER (WHERE has_outcome_code) / count(*), 1) AS outcome_coded_pct,
           count(*) FILTER (WHERE has_death_outcome) AS death_outcome_cases,
           round(100.0 * count(*) FILTER (WHERE has_death_outcome) / count(*), 1) AS death_outcome_pct,
           count(*) FILTER (WHERE is_provisional) AS provisional_cases,
           round(100.0 * count(*) FILTER (WHERE is_provisional) / count(*), 1) AS provisional_pct,
           round(100.0 * max(concentration.top10_cases) / max(concentration.all_ps_cases), 1) AS top10_ps_drug_pct
    FROM clean_case CROSS JOIN concentration
""").df()
display(landscape_kpis)

### 12b. Signal Workbench: observed drug–reaction case counts

This table gives distinct reported cases per primary-suspect ingredient and reaction. It is **not** PRR, chi-square, or proof of a signal. Those statistics require a defined comparison population and a separate validated calculation; the Stage 2 database does not contain them yet. The candidate-reduced count is a sensitivity column, not the default count.

In [ ]:
MIN_PAIR_CASES = 3
display(con.execute("""
    SELECT c.ps_ingredient_std AS ingredient, r.pt AS reaction,
           count(DISTINCT c.caseid) AS reported_cases,
           count(DISTINCT c.caseid) FILTER (WHERE c.in_duplicate_reduced_set) AS cases_in_duplicate_reduced_set,
           round(100.0 * count(DISTINCT c.caseid) FILTER (WHERE c.has_outcome_code)
                 / count(DISTINCT c.caseid), 1) AS outcome_coded_pct
    FROM clean_case c
    JOIN clean_reaction r USING (primaryid)
    WHERE c.ps_status = 'PS' AND c.ps_ingredient_std IS NOT NULL
    GROUP BY 1, 2
    HAVING count(DISTINCT c.caseid) >= ?
    ORDER BY reported_cases DESC, ingredient, reaction
    LIMIT 25
""", [MIN_PAIR_CASES]).df())

### 12c. Drug Profile: choose a primary-suspect ingredient

The first query provides case-level cards for one exact `ps_ingredient_std` value. The next shows its most commonly reported reactions. Change the selector to another value from the top-drug list. These are reported-case counts; the comparison to all other drugs is descriptive, not an exposure-adjusted risk comparison.

In [ ]:
DRUG_TO_EXPLORE = con.execute("""
    SELECT ps_ingredient_std FROM clean_case
    WHERE ps_status = 'PS' AND ps_ingredient_std IS NOT NULL
    GROUP BY 1 ORDER BY count(*) DESC LIMIT 1
""").fetchone()[0]
print("Selected ingredient:", DRUG_TO_EXPLORE)

display(con.execute("""
    SELECT
      count(*) FILTER (WHERE ps_ingredient_std = ?) AS drug_cases,
      count(*) FILTER (WHERE ps_ingredient_std = ? AND has_outcome_code) AS drug_outcome_cases,
      round(100.0 * count(*) FILTER (WHERE ps_ingredient_std = ? AND has_outcome_code)
            / nullif(count(*) FILTER (WHERE ps_ingredient_std = ?), 0), 1) AS drug_outcome_pct,
      count(*) FILTER (WHERE ps_ingredient_std = ? AND has_death_outcome) AS drug_death_cases,
      median(age_years) FILTER (WHERE ps_ingredient_std = ?) AS drug_median_age,
      round(100.0 * count(*) FILTER (WHERE ps_ingredient_std = ? AND sex_std = 'Female')
            / nullif(count(*) FILTER (WHERE ps_ingredient_std = ? AND sex_std IN ('Female','Male')), 0), 1) AS known_sex_female_pct,
      count(*) FILTER (WHERE ps_ingredient_std = ? AND is_provisional) AS drug_provisional_cases,
      count(*) AS all_cases,
      count(*) FILTER (WHERE has_outcome_code) AS all_outcome_cases
    FROM clean_case
""", [DRUG_TO_EXPLORE] * 9).df())

display(con.execute("""
    SELECT r.pt AS reaction, count(DISTINCT c.caseid) AS cases
    FROM clean_case c JOIN clean_reaction r USING (primaryid)
    WHERE c.ps_ingredient_std = ? AND c.ps_status = 'PS'
    GROUP BY 1 ORDER BY cases DESC, reaction LIMIT 15
""", [DRUG_TO_EXPLORE]).df())

### 12d. Patient Population: age and sex mix

Unknown age/sex categories remain visible. `age_cohort` gives the broad pediatric/adult/elderly bands; `age_band_order` supplies chart ordering. The wireframe's therapeutic-area charts cannot be produced until indications are mapped to reviewed therapeutic areas.

In [ ]:
display(con.execute("""
    SELECT age_band, age_band_order, sex_std, count(*) AS cases
    FROM clean_case
    GROUP BY 1, 2, 3
    ORDER BY age_band_order NULLS LAST, sex_std
""").df())

print("Primary-suspect case mix by age cohort, top drugs:")
display(con.execute("""
    SELECT ps_ingredient_std AS ingredient, age_cohort, count(*) AS cases
    FROM clean_case
    WHERE ps_status = 'PS' AND ps_ingredient_std IS NOT NULL
    GROUP BY 1, 2
    QUALIFY dense_rank() OVER (PARTITION BY age_cohort ORDER BY count(*) DESC) <= 5
    ORDER BY age_cohort, cases DESC, ingredient
""").df())

### 12e. Reporting and Data Health: completeness by received quarter

This tests the requested completeness indicators and keeps provisional quarters visible. Follow-up maturity is only approximated here by whether the selected latest `caseversion` is greater than 1; exact first-follow-up timing by initial cohort needs version-history data. Reporting lag can be explored for valid event dates, but later-reported follow-up events may be after the initial receipt date.

In [ ]:
display(con.execute("""
    SELECT first_received_quarter_start, first_received_quarter,
           is_provisional, count(*) AS cases,
           round(100.0 * count(*) FILTER (WHERE age_status = 'ok') / count(*), 1) AS age_known_pct,
           round(100.0 * count(*) FILTER (WHERE sex_std IN ('Female','Male')) / count(*), 1) AS sex_known_pct,
           round(100.0 * count(*) FILTER (WHERE event_date_status = 'ok') / count(*), 1) AS event_date_valid_pct,
           round(100.0 * count(*) FILTER (WHERE reporter_type <> 'Not reported') / count(*), 1) AS occupation_reported_pct,
           round(100.0 * count(*) FILTER (WHERE caseversion > 1) / count(*), 1) AS version_gt1_proxy_pct
    FROM clean_case
    WHERE first_received_cohort = 'In window'
    GROUP BY 1, 2, 3 ORDER BY 1
""").df())

print("Onset data coverage (usable onset is the denominator to report for onset charts):")
display(con.execute("""
    SELECT onset_status, count(*) AS cases,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct_of_onset_rows
    FROM clean_onset GROUP BY 1 ORDER BY cases DESC
""").df())

print("Country codes for map review; do not assume region/unknown codes are countries:")
display(con.execute("""
    SELECT geo_country_code, geo_country_source, occr_country_status, count(*) AS cases
    FROM clean_case GROUP BY 1, 2, 3 ORDER BY cases DESC NULLS LAST LIMIT 15
""").df())

print("Reporting lag for valid event dates (negative values can reflect later-version reports):")
display(con.execute("""
    WITH lags AS (
      SELECT date_diff('day', event_date, first_received_date) AS lag_days
      FROM clean_case WHERE event_date_status = 'ok' AND first_received_date IS NOT NULL
    )
    SELECT CASE WHEN lag_days < 0 THEN 'Negative (event after first receipt)'
                WHEN lag_days <= 7 THEN '0-7 days' WHEN lag_days <= 30 THEN '8-30 days'
                WHEN lag_days <= 90 THEN '31-90 days' WHEN lag_days <= 365 THEN '91-365 days'
                ELSE 'Over 1 year' END AS lag_bucket,
           count(*) AS cases, min(lag_days) AS min_days, max(lag_days) AS max_days
    FROM lags GROUP BY 1 ORDER BY min(lag_days)
""").df())

## 13. Read quality flags instead of hiding data issues

Cleaning does not mean every case has every field. The status fields tell you why a value may be unavailable, and duplicate matches are candidates rather than confirmed duplicates. Try changing `STATUS_COLUMN` below to `wt_status`, `event_date_status`, `duplicate_check_status`, or `ps_status`.

In [ ]:
STATUS_COLUMN = "age_status"
valid_case_columns = {
    row[0] for row in con.execute("""
        SELECT column_name FROM information_schema.columns
        WHERE table_catalog = current_database() AND table_schema = 'main'
          AND table_name = 'clean_case'
    """).fetchall()
}
if STATUS_COLUMN not in valid_case_columns:
    raise ValueError(f"Unknown clean_case column: {STATUS_COLUMN!r}")
status_sql = '"' + STATUS_COLUMN.replace('"', '""') + '"'
display(con.execute(
    f'SELECT {status_sql} AS status, count(*) AS cases '
    f'FROM clean_case GROUP BY 1 ORDER BY cases DESC NULLS LAST'
).df())

## 14. Your SQL playground

Edit the query to test a question from one of your dashboard wireframes. Keep a `LIMIT` while exploring. For case counts from a child table, use `count(DISTINCT caseid)` unless you have deliberately aggregated to one row per case first.

In [ ]:
SQL_TO_TRY = """
SELECT age_band, count(*) AS cases
FROM clean_case
GROUP BY age_band
ORDER BY cases DESC NULLS LAST
LIMIT 20
"""
display(con.execute(SQL_TO_TRY).df())

## 15. Interpretation reminders

- Join child tables to `clean_case` using `primaryid`. Use both `primaryid` and `drug_seq_raw` to link drug-level dose and indication details.
- Pre-aggregate one-to-many tables before joining multiple children, or row multiplication can inflate counts.
- Use `first_received_quarter` for new-report trends; `source_quarter` means the file containing the selected latest version.
- Treat `is_provisional`, missingness statuses, and duplicate-candidate flags as visible data context.
- FAERS reports describe suspected adverse events. They do not prove causation and have no exposure denominator for incidence calculations.
- Country and parent-ingredient mapping remain separate review decisions; don't treat suggested drug mappings as approved ground truth.

## 16. Close the connection

Run this cell when you are finished. It only closes the notebook's read-only connection.

In [ ]:
con.close()
print("Read-only DuckDB connection closed.")